# Pipeline Verification

Checks that `src/preprocess.py` (used by the Streamlit app for raw uploads and single predictions) reproduces the training features exactly.

The test rebuilds `X_test` from the raw CSV using the app's pipeline and compares it to the saved `X_test.pkl`. Any mismatch would make the app predict silently wrong values without raising an error.

## Load Artifacts

In [ ]:
import sys
import pickle
import pandas as pd

sys.path.append('../src')
from preprocess import preprocess_raw, validate_raw

with open('../data/processed/splits/X_test.pkl', 'rb') as f:
    X_test = pickle.load(f)
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

raw = pd.read_csv('../data/raw/telco_customer_churn.csv')
print(f"Raw data: {raw.shape}")
print(f"X_test: {X_test.shape}")

## Rebuild X_test from Raw Data

`X_test` keeps the original row index from the raw CSV (no rows were dropped during preprocessing), so the same customers can be selected with `.loc`.

In [ ]:
raw_test = raw.loc[X_test.index]
rebuilt = preprocess_raw(raw_test, scaler, X_test.columns)

print(f"Rebuilt shape: {rebuilt.shape}")
print(f"Column order identical: {list(rebuilt.columns) == list(X_test.columns)}")

## Compare

In [ ]:
max_diff = (rebuilt.astype(float) - X_test.astype(float)).abs().max()

print(f"Rows compared: {len(rebuilt)}")
print(f"Largest difference across all columns: {max_diff.max():.2e}")

if max_diff.max() < 1e-9:
    print("PASS: app preprocessing is identical to the notebook pipeline")
else:
    print("FAIL: columns with differences:")
    print(max_diff[max_diff > 1e-9])

## Validation Check

Confirms that the raw data passes the app's input validation, and that a typo is rejected with a clear message.

In [ ]:
print("Raw test set validation errors:", validate_raw(raw_test) or "none")

broken = raw_test.head(3).copy()
broken.iloc[0, broken.columns.get_loc('InternetService')] = 'fiber optic'
print("\nTypo example:")
for err in validate_raw(broken):
    print(f"- {err}")